In [0]:
%run ./test

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, sum, count, round

In [0]:
absence_df = spark.read.table("nba_insights.silver.games_stats_with_stars_absence")
franchise_history_df = spark.read.table("nba_insights.silver.star_franchise_games_history") 
stars_df = spark.read.table("nba_insights.silver.star_players")

In [0]:
print(f"Absence table count: {absence_df.count()}")
print(f"Franchise history count: {franchise_history_df.count()}")
print(f"Stars count: {stars_df.count()}")

In [0]:
def determine_absence_outcomes(absence_df: DataFrame) -> DataFrame:
    """Evaluates whether the star's team won or lost during their absence."""
    return absence_df.withColumn(
        "team_outcome",
        when(col("star_team_id") == col("team_id_home"), col("wl_home"))
        .when(col("star_team_id") == col("team_id_away"), col("wl_away"))
        .otherwise(None)
    )

outcomes_df = determine_absence_outcomes(absence_df)
display(outcomes_df)

In [0]:
suite = unittest.TestSuite()
suite.addTest(TestPlayerPipeline('test_determine_absence_outcomes'))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [0]:
def aggregate_absence_stats(outcomes_df: DataFrame) -> DataFrame:
    """Calculates total games missed and wins during those absences."""
    return outcomes_df.groupBy("player_id").agg(
        count("game_id").alias("games_absent"),
        sum(when(col("team_outcome") == "W", 1).otherwise(0)).alias("absence_wins")
    )

absence_agg_df = aggregate_absence_stats(outcomes_df)
display(absence_agg_df)

In [0]:
suite = unittest.TestSuite()
suite.addTest(TestPlayerPipeline('test_aggregate_absence_stats'))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [0]:
def aggregate_franchise_stats(franchise_history_df: DataFrame) -> DataFrame:
    """Calculates total games and wins for the star's franchises during their era."""
    return franchise_history_df.groupBy("player_id").agg(
        count("game_id").alias("total_team_games"),
        sum(when(col("team_outcome") == "W", 1).otherwise(0)).alias("total_team_wins")
    )

totals_agg_df = aggregate_franchise_stats(franchise_history_df)
display(totals_agg_df)

In [0]:
suite = unittest.TestSuite()
suite.addTest(TestPlayerPipeline('test_aggregate_franchise_stats'))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [0]:
def aggregate_franchise_stats(franchise_history_df: DataFrame) -> DataFrame:
    """Calculates total games and wins for the star's franchises during their era."""
    return franchise_history_df.groupBy("player_id").agg(
        count("game_id").alias("total_team_games"),
        sum(when(col("team_outcome") == "W", 1).otherwise(0)).alias("total_team_wins")
    )

totals_agg_df = aggregate_franchise_stats(franchise_history_df)
display(totals_agg_df)

In [0]:
suite = unittest.TestSuite()
suite.addTest(TestPlayerPipeline('test_calculate_impact_delta'))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [0]:
def calculate_impact_delta(totals_agg_df: DataFrame, absence_agg_df: DataFrame) -> DataFrame:
    """Deducts absences from totals to calculate baseline stats and the final Win % Delta."""
    delta_df = totals_agg_df.join(absence_agg_df, on="player_id", how="left").fillna(0)

    final_math_df = delta_df.withColumn(
        "games_present", col("total_team_games") - col("games_absent")
    ).withColumn(
        "presence_wins", col("total_team_wins") - col("absence_wins")
    )

    return final_math_df.withColumn(
        "win_pct_with_star", round(col("presence_wins") / col("games_present"), 3)
    ).withColumn(
        "win_pct_without_star", 
        when(col("games_absent") > 0, round(col("absence_wins") / col("games_absent"), 3)).otherwise(0.0)
    ).withColumn(
        "impact_delta", round(col("win_pct_with_star") - col("win_pct_without_star"), 3)
    )

delta_df = calculate_impact_delta(totals_agg_df, absence_agg_df)
display(delta_df)

In [0]:
suite = unittest.TestSuite()
suite.addTest(TestPlayerPipeline('test_format_gold_summary'))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [0]:
def format_gold_summary(delta_df: DataFrame, stars_df: DataFrame) -> DataFrame:
    """Joins player names and selects the final Gold columns."""
    return delta_df.join(
        stars_df.select("player_id", "display_first_last"), 
        on="player_id"
    ).select(
        "player_id", 
        "display_first_last",
        "games_present", 
        "win_pct_with_star",
        "games_absent", 
        "win_pct_without_star",
        "impact_delta"
    )

gold_star_impact = format_gold_summary(delta_df, stars_df)
display(gold_star_impact)

In [0]:
def process_gold_star_impact(absence_df: DataFrame, franchise_history_df: DataFrame, stars_df: DataFrame) -> DataFrame:
    """Orchestrates the Gold transformations end-to-end."""
    out_df = determine_absence_outcomes(absence_df)
    abs_agg_df = aggregate_absence_stats(out_df)
    tot_agg_df = aggregate_franchise_stats(franchise_history_df)
    math_df = calculate_impact_delta(tot_agg_df, abs_agg_df)
    return format_gold_summary(math_df, stars_df)

final_gold_df = process_gold_star_impact(absence_df, franchise_history_df, stars_df)
display(final_gold_df)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


gold_pd = final_gold_df.toPandas()

gold_pd = gold_pd.sort_values('impact_delta', ascending=False)
top_stars = gold_pd.head(15)

print(top_stars[['display_first_last', 'impact_delta']])

In [0]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_stars, 
    x='impact_delta', 
    y='display_first_last', 
    palette='viridis'
)

plt.title("The Superstar Impact: Win Percentage Delta (Top 15)", fontsize=14, fontweight='bold')
plt.xlabel("Win % Delta (With Star - Without Star)", fontsize=12)
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.show()

In [0]:
x = np.arange(len(top_stars['display_first_last']))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, top_stars['win_pct_with_star'], width, label='With Star', color='#1f77b4')
rects2 = ax.bar(x + width/2, top_stars['win_pct_without_star'], width, label='Without Star', color='#ff7f0e')

ax.set_ylabel('Win Percentage', fontsize=12)
ax.set_title('Team Win Percentage: With vs. Without Superstar', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top_stars['display_first_last'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=gold_pd, 
    x='games_absent', 
    y='impact_delta', 
    size='games_present',
    sizes=(50, 400),
    alpha=0.6,
    color='#2ca02c'
)


plt.axhline(0, color='red', linestyle='--')
plt.title("Statistical Confidence: Impact vs. Games Missed", fontsize=14, fontweight='bold')
plt.xlabel("Total Games Absent (Sample Size)", fontsize=12)
plt.ylabel("Win % Impact Delta", fontsize=12)

for idx, row in gold_pd.nlargest(3, 'impact_delta').iterrows():
    plt.text(row['games_absent'] + 2, row['impact_delta'], row['display_first_last'], fontsize=6)

plt.grid(True, linestyle='--', alpha=0.5)
plt.show()